<a href="https://colab.research.google.com/github/Crtljuliette/C-Miniprojet/blob/main/Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =============================================================================
# TRANSFORMER PINN — Cylindre de l'article (data-driven solution)
# Comparaison avec le MLP de Raissi
#
# Approche : solution (ν=0.01 fixé, λ1=1 fixé)
# Entrées   : (x, y, t)
# Sorties   : (u, v, p)
# Données   : cylinder_nektar_wake.mat
# =============================================================================

import numpy as np
import scipy.io
import torch
import torch.nn as nn
import torch.optim as optim
import torch.autograd as autograd
import matplotlib.pyplot as plt
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

NU = 0.01   # viscosité fixée comme dans l'article

# =============================================================================
# 1. CHARGEMENT DES DONNÉES (identique à l'article)
# =============================================================================
data   = scipy.io.loadmat('/content/cylinder_nektar_wake.mat')
U_star = data['U_star']   # (N, 2, T)
P_star = data['p_star']   # (N, T)
t_star = data['t']        # (T, 1)
X_star = data['X_star']   # (N, 2)

N = X_star.shape[0]
T = t_star.shape[0]

XX = np.tile(X_star[:, 0:1], (1, T))
YY = np.tile(X_star[:, 1:2], (1, T))
TT = np.tile(t_star, (1, N)).T
UU = U_star[:, 0, :]
VV = U_star[:, 1, :]
PP = P_star

x = XX.flatten()[:, None]
y = YY.flatten()[:, None]
t = TT.flatten()[:, None]
u = UU.flatten()[:, None]
v = VV.flatten()[:, None]
p = PP.flatten()[:, None]

print(f"Total points : {x.shape[0]:,}  ({N} spatial × {T} time steps)")

# Échantillonnage pour l'entraînement
N_TRAIN = 5000
np.random.seed(42); torch.manual_seed(42)
idx = np.random.choice(N*T, N_TRAIN, replace=False)

def T_ten(arr, g=False):
    return torch.tensor(arr, dtype=torch.float32,
                        requires_grad=g).to(device)

x_tr = T_ten(x[idx], True);  y_tr = T_ten(y[idx], True)
t_tr = T_ten(t[idx], True);  u_tr = T_ten(u[idx])
v_tr = T_ten(v[idx])

# =============================================================================
# 2. MLP — architecture Raissi originale
#    Entrée (x,y,t) → [20]×8 → (u,v,p)
# =============================================================================
class MLP_PINN(nn.Module):
    def __init__(self, layers=[3,20,20,20,20,20,20,20,20,3]):
        super().__init__()
        mods = []
        for i in range(len(layers)-2):
            mods += [nn.Linear(layers[i], layers[i+1]), nn.Tanh()]
        mods += [nn.Linear(layers[-2], layers[-1])]
        self.net = nn.Sequential(*mods)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x, y, t):
        inp = torch.cat([x, y, t], dim=1)
        out = self.net(inp)
        return out[:,0:1], out[:,1:2], out[:,2:3]   # u, v, p

# =============================================================================
# 3. TRANSFORMER PINN
#
# Architecture :
#   (x,y,t) → Embedding (3→d_model)
#           → N blocs Transformer
#              ├─ Multi-head Self-Attention
#              ├─ Layer Norm + résidu
#              ├─ Feed-Forward (d_model → d_ff → d_model)
#              └─ Layer Norm + résidu
#           → Output head (d_model→3) → (u,v,p)
#
# Différence clé avec MLP :
#   MLP  : traite chaque point indépendamment
#   Transf : chaque point "voit" tous les autres via l'attention
# =============================================================================
class TransformerBlock(nn.Module):
    """
    Un bloc Transformer = Attention + Feed-Forward + résidus.

    L'attention calcule pour chaque point un score avec tous les autres :
      Attention(Q,K,V) = softmax(QKᵀ / √d_k) × V
    où Q=K=V=entrée (self-attention).
    """
    def __init__(self, d_model=64, n_heads=4, d_ff=256, dropout=0.0):
        super().__init__()
        # Multi-head self-attention
        # n_heads têtes en parallèle, chacune capture un type de relation
        self.attn    = nn.MultiheadAttention(d_model, n_heads,
                                              dropout=dropout,
                                              batch_first=True)
        self.norm1   = nn.LayerNorm(d_model)

        # Feed-forward local (appliqué point par point après attention)
        self.ff      = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),             # GELU plus doux que ReLU, standard en Transformer
            nn.Linear(d_ff, d_model)
        )
        self.norm2   = nn.LayerNorm(d_model)

    def forward(self, x):
        # Self-attention + résidu
        # x shape : (batch, seq_len, d_model)
        attn_out, _ = self.attn(x, x, x)
        x = self.norm1(x + attn_out)    # connexion résiduelle

        # Feed-forward + résidu
        x = self.norm2(x + self.ff(x))
        return x


class Transformer_PINN(nn.Module):
    """
    PINN basé sur un Transformer.

    Points importants :
    - Chaque point (x,y,t) est un "token" de la séquence
    - L'attention permet à chaque point de dépendre des autres
    - En CFD c'est physiquement motivé : u(x) dépend du voisinage
    - On garde la même formulation physique (résidus NS) que le MLP
    """
    def __init__(self, d_model=64, n_heads=4, n_layers=4, d_ff=256):
        super().__init__()

        # Embedding : projette (x,y,t) → d_model dimensions
        self.embedding = nn.Sequential(
            nn.Linear(3, d_model),
            nn.Tanh()
        )

        # Blocs Transformer empilés
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff)
            for _ in range(n_layers)
        ])

        # Tête de sortie : d_model → (u,v,p)
        self.head = nn.Linear(d_model, 3)

        # Initialisation
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x, y, t):
        # Entrée : N points × 3 coordonnées
        inp = torch.cat([x, y, t], dim=1)   # (N, 3)

        # Pour l'attention on a besoin d'une dimension séquence
        # On traite les N points comme une séquence de N tokens
        inp = inp.unsqueeze(0)              # (1, N, 3)  batch=1, seq=N

        # Embedding
        h = self.embedding(inp)             # (1, N, d_model)

        # Blocs Transformer
        for block in self.blocks:
            h = block(h)                    # (1, N, d_model)

        # Sortie
        out = self.head(h)                  # (1, N, 3)
        out = out.squeeze(0)                # (N, 3)

        return out[:,0:1], out[:,1:2], out[:,2:3]   # u, v, p

# =============================================================================
# 4. RÉSIDUS NS (identique pour MLP et Transformer)
# =============================================================================
def G(f, x): return autograd.grad(f.sum(), x, create_graph=True)[0]

def ns_residuals(model, x, y, t):
    u, v, p = model(x, y, t)

    u_t=G(u,t); u_x=G(u,x); u_y=G(u,y)
    v_t=G(v,t); v_x=G(v,x); v_y=G(v,y)
    p_x=G(p,x); p_y=G(p,y)
    u_xx=G(u_x,x); u_yy=G(u_y,y)
    v_xx=G(v_x,x); v_yy=G(v_y,y)

    f_u = u_t + u*u_x + v*u_y + p_x - NU*(u_xx+u_yy)
    f_v = v_t + u*v_x + v*v_y + p_y - NU*(v_xx+v_yy)
    f_c = u_x + v_y   # continuité

    return f_u, f_v, f_c, u, v

def compute_loss(model, x, y, t, u_true, v_true):
    f_u, f_v, f_c, u_pred, v_pred = ns_residuals(model, x, y, t)

    loss_dat = ((u_true-u_pred)**2).mean() + ((v_true-v_pred)**2).mean()
    loss_pde = (f_u**2).mean() + (f_v**2).mean() + (f_c**2).mean()

    return loss_dat + loss_pde, loss_dat, loss_pde

# =============================================================================
# 5. ENTRAÎNEMENT GÉNÉRIQUE
# =============================================================================
def train_model(model, name, adam_iters=2000, lbfgs_iters=10000):
    model.to(device)
    print(f"\n{'='*60}")
    print(f"Entraînement : {name}")
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Paramètres   : {n_params:,}")
    print(f"{'='*60}")

    history = {'total':[], 'dat':[], 'pde':[]}

    # ── Adam ──────────────────────────────────────────────────────────────
    opt = optim.Adam(model.parameters(), lr=1e-3)
    scheduler = optim.lr_scheduler.StepLR(opt, step_size=500, gamma=0.5)
    t0 = time.time()

    for it in range(adam_iters):
        opt.zero_grad()
        total, ld, lp = compute_loss(model, x_tr, y_tr, t_tr, u_tr, v_tr)
        total.backward()
        opt.step()
        scheduler.step()

        history['total'].append(total.item())
        history['dat'].append(ld.item())
        history['pde'].append(lp.item())

        if it % 200 == 0:
            print(f"  Adam {it:5d} | tot={total.item():.3e} "
                  f"dat={ld.item():.3e} pde={lp.item():.3e} | "
                  f"{time.time()-t0:.1f}s")
            t0=time.time()

    # ── L-BFGS ────────────────────────────────────────────────────────────
    print(f"\n  L-BFGS...")
    opt_lb = optim.LBFGS(model.parameters(), max_iter=lbfgs_iters,
                         tolerance_grad=1e-9, tolerance_change=1e-9,
                         history_size=50, line_search_fn="strong_wolfe")
    cpt=[0]

    def closure():
        opt_lb.zero_grad()
        total, ld, lp = compute_loss(model, x_tr, y_tr, t_tr, u_tr, v_tr)
        total.backward()
        history['total'].append(total.item())
        cpt[0]+=1
        if cpt[0]%200==0:
            print(f"  LBFGS {cpt[0]:5d} | tot={total.item():.3e} "
                  f"dat={ld.item():.3e} pde={lp.item():.3e}")
        return total

    opt_lb.step(closure)

    return history

# =============================================================================
# 6. ENTRAÎNEMENT ET COMPARAISON
# =============================================================================
# MLP (Raissi original)
mlp   = MLP_PINN()
hist_mlp = train_model(mlp, "MLP Raissi", adam_iters=2000, lbfgs_iters=5000)
torch.save(mlp.state_dict(), '/content/mlp_cylindre.pt')

# Transformer
transf = Transformer_PINN(d_model=64, n_heads=4, n_layers=4, d_ff=256)
hist_transf = train_model(transf, "Transformer", adam_iters=2000, lbfgs_iters=5000)
torch.save(transf.state_dict(), '/content/transformer_cylindre.pt')

# =============================================================================
# 7. ÉVALUATION — snapshot 100
# =============================================================================
SNAP = 100
x_test = T_ten(X_star[:,0:1], True).to(device)
y_test = T_ten(X_star[:,1:2], True).to(device)
t_test = T_ten(np.full((N,1), t_star[SNAP,0]), True).to(device)

u_ref  = U_star[:,0,SNAP]
v_ref  = U_star[:,1,SNAP]
p_ref  = P_star[:,SNAP]

def evaluate(model, name):
    model.eval()
    u_p, v_p, p_p = model(x_test, y_test, t_test)
    u_p = u_p.detach().cpu().numpy().flatten()
    v_p = v_p.detach().cpu().numpy().flatten()
    p_p = p_p.detach().cpu().numpy().flatten()

    err_u = np.linalg.norm(u_ref-u_p) / np.linalg.norm(u_ref)
    err_v = np.linalg.norm(v_ref-v_p) / np.linalg.norm(v_ref)
    err_p = np.linalg.norm(p_ref-p_p) / np.linalg.norm(p_ref)

    print(f"\n{name} :")
    print(f"  Erreur u : {err_u:.4e}")
    print(f"  Erreur v : {err_v:.4e}")
    print(f"  Erreur p : {err_p:.4e}  ← reconstruit sans données p !")
    return u_p, v_p, p_p, err_u, err_v, err_p

print(f"\n{'='*60}\nÉVALUATION (snapshot t={t_star[SNAP,0]:.2f}s)\n{'='*60}")
u_mlp,   v_mlp,   p_mlp,   eu_m, ev_m, ep_m = evaluate(mlp,   "MLP Raissi")
u_tran,  v_tran,  p_tran,  eu_t, ev_t, ep_t = evaluate(transf,"Transformer")

# =============================================================================
# 8. VISUALISATION — comparaison côte à côte
# =============================================================================
from scipy.interpolate import griddata

lb = X_star.min(0);  ub = X_star.max(0)
n_grid = 150
xg = np.linspace(lb[0], ub[0], n_grid)
yg = np.linspace(lb[1], ub[1], n_grid)
Xg, Yg = np.meshgrid(xg, yg)

def interp(vals):
    return griddata(X_star, vals, (Xg, Yg), method='cubic')

# Interpolation sur grille
fields = {
    'u_ref' : interp(u_ref),
    'u_mlp' : interp(u_mlp),
    'u_tran': interp(u_tran),
    'p_ref' : interp(p_ref),
    'p_mlp' : interp(p_mlp),
    'p_tran': interp(p_tran),
}

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
fig.suptitle(f'Comparaison MLP vs Transformer  (t={t_star[SNAP,0]:.2f}s)\n'
             f'MLP  → err_u={eu_m:.2e}  err_p={ep_m:.2e}\n'
             f'Transf → err_u={eu_t:.2e}  err_p={ep_t:.2e}',
             fontsize=12)

rows = [
    (fields['u_ref'],  fields['p_ref'],  'Référence CFD'),
    (fields['u_mlp'],  fields['p_mlp'],  f'MLP Raissi  (err_u={eu_m:.2e})'),
    (fields['u_tran'], fields['p_tran'], f'Transformer (err_u={eu_t:.2e})'),
]

for i, (u_f, p_f, title) in enumerate(rows):
    for j, (field, label, cmap) in enumerate([
            (u_f, 'u [m/s]', 'RdBu_r'),
            (p_f, 'p [Pa]',  'RdBu_r')]):
        ax = axes[i,j]
        cf = ax.contourf(Xg, Yg, field, levels=50, cmap=cmap)
        plt.colorbar(cf, ax=ax, shrink=0.8)
        ax.set_title(f'{title} — {label}', fontsize=10)
        ax.set_xlabel('x'); ax.set_ylabel('y')
        ax.set_aspect('equal')

plt.tight_layout()
plt.savefig('/content/comparaison_mlp_transformer.png', dpi=150, bbox_inches='tight')
plt.show()

# Courbes de convergence
fig2, ax2 = plt.subplots(figsize=(11,4))
ax2.semilogy(hist_mlp['total'],   'b-',  lw=1.5, label='MLP — total')
ax2.semilogy(hist_mlp['dat'],     'b--', lw=1.0, label='MLP — données')
ax2.semilogy(hist_transf['total'],'r-',  lw=1.5, label='Transformer — total')
ax2.semilogy(hist_transf['dat'],  'r--', lw=1.0, label='Transformer — données')
ax2.set_xlabel('Itération'); ax2.set_ylabel('Loss')
ax2.set_title('Convergence : MLP vs Transformer')
ax2.legend(); ax2.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig('/content/convergence_comparaison.png', dpi=150, bbox_inches='tight')
plt.show()

# Tableau récapitulatif
print(f"\n{'='*60}")
print("RÉCAPITULATIF COMPARAISON")
print(f"{'='*60}")
print(f"{'Modèle':<15} {'Params':>8} {'Err u':>10} {'Err v':>10} {'Err p':>10}")
print(f"{'-'*55}")
print(f"{'MLP Raissi':<15} "
      f"{sum(p.numel() for p in mlp.parameters()):>8,} "
      f"{eu_m:>10.4e} {ev_m:>10.4e} {ep_m:>10.4e}")
print(f"{'Transformer':<15} "
      f"{sum(p.numel() for p in transf.parameters()):>8,} "
      f"{eu_t:>10.4e} {ev_t:>10.4e} {ep_t:>10.4e}")
print(f"\n  Note : p est reconstruit sans données de pression (via NS uniquement)")

Device : cpu
Total points : 1,000,000  (5000 spatial × 200 time steps)

Entraînement : MLP Raissi
Paramètres   : 3,083
  Adam     0 | tot=6.513e-01 dat=6.390e-01 pde=1.229e-02 | 0.4s


KeyboardInterrupt: 